<a href="https://colab.research.google.com/github/jdasam/aat3020/blob/2025/notebooks/3_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch as th
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


# Language modeling

In [ ]:
#!wget "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"

In [ ]:
def read_txt(txt_path):
  with open(txt_path, 'r') as f:
    txt_string = f.readlines()
  return txt_string

txt_string = read_txt('data_3/names.txt')

In [ ]:
names_list = [x.replace('\n', '') for x in txt_string]
len(names_list)

In [ ]:
names_list

# N-Gram
- Start with bi-gram (2-gram)

In [ ]:
from collections import defaultdict

# bigram_dict = {}
bigram_dict = defaultdict(int) # If key is not in the defaultdict, it automatically assign key and empty value (int=0, list=[])
unigram_dict = defaultdict(int)

# RNN
- $h_t = \tanh(\textbf{W}_{hh}h_{t-1} + \textbf{W}_{xh}x_t + b) $
  - $\textbf{W}$: Weight Matrix
  - $b$: bias
  - $x_t$: input vector of time step $t$
  - $h_t$: hidden state (and also output) of time step $t$


In [ ]:
torch.manual_seed(0)
sequence_length = 7
input_dim, hidden_dim = 3, 5
weight_hh = nn.Linear(hidden_dim, hidden_dim)
weight_xh = nn.Linear(input_dim, hidden_dim)
h0 = torch.zeros(hidden_dim)
x = torch.randn([sequence_length, input_dim])
t = 0
x_t = x[t]
x[t]

In [ ]:
h_t = torch.tanh(weight_hh(h0)+weight_xh(x_t))
h_t

In [ ]:
def run_rnn_cell(weight_hh, weight_xh, prev_h, x_t):
  return torch.tanh(weight_hh(prev_h)+weight_xh(x_t))

output = []
prev_h = h0
for i in range(len(x)):
  print(f"x: {x[i]}")
  h = run_rnn_cell(weight_hh, weight_xh, prev_h, x[i])
  prev_h = h
  output.append(h)
  print(f"h: {h}")

output = torch.stack(output)
output

In [ ]:
names_list[:10]

In [ ]:
entire_chars = []

for name in names_list:
  for char in name:
    entire_chars.append(char)

len(entire_chars)

In [ ]:
set(entire_chars)
vocab = list(set(entire_chars))
vocab.sort()

char2idx = {char: i for i, char in enumerate(vocab)}
char2idx

## Define Dataset Class

In [ ]:
class Nameset:
  def __init__ (self, text_fn):
    self.data = [x.replace('\n', '') for x in read_txt(text_fn)]
    entire_chars = [char for name in self.data for char in name]
    self.vocab = list(set(entire_chars))
    self.vocab.sort()
    self.vocab = ['<pad>', '<start>', '<end>'] + self.vocab
    self.char2idx = {char: i for i, char in enumerate(self.vocab)}

  def read_txt(self, txt_path):
    with open(txt_path, 'r') as f:
      txt_string = f.readlines()
    return txt_string

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    model_input = self.word2idx(self.data[idx])[:-1]
    model_target = self.word2idx(self.data[idx])[1:]
    return model_input, model_target

  def word2idx(self, word):
    return [self.char2idx['<start>']] + \
           [self.char2idx[char] for char in word] + \
           [self.char2idx['<end>']]

dataset = Nameset('data_3/names.txt')

dataset[0]

## Define the model

In [ ]:
th.manual_seed(0)

class LanguageModel(nn.Module):
  def __init__(self, vocab_size, embedding_dim=16):
    super().__init__()
    self.vocab_size = vocab_size
    self.emb = nn.Embedding(embedding_dim=embedding_dim, num_embeddings=vocab_size)
    self.rnn = nn.GRU(input_size=embedding_dim, hidden_size=2*embedding_dim)
    self.proj = nn.Linear(in_features=2*embedding_dim, out_features=vocab_size)

  def forward(self, x):
    x, last_h = self.rnn(self.emb(x))
    x = self.proj(x)
    return torch.softmax(x, dim=1)

vocab_size = len(dataset.vocab)
model = LanguageModel(vocab_size)
model.emb.weight
x,y = dataset[0]
x = torch.tensor(x)
print(x)
print(model(x).shape)
model(x)

## Define Collate Function
- As the input is a list of arbitrary length, we need to pad them to the same length\n
- You can feed collate function to the DataLoader
- ```dataloader = DataLoader(dataset, batch_size=10, collate_fn=collate_fn)```


## Define the split function
- Split the dataset into training and validation set
- You can use ```torch.utils.data.random_split``` function
- ```train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])```

## Define Training Loop

In [ ]:
x, y = dataset[0]
x = torch.tensor(x)
y = torch.tensor(y)
pred = model(x)

prob_of_correct_char = pred[torch.arange(len(y)), y]
nll = -torch.log(prob_of_correct_char)
nll.mean()


## Define the Inference
- Unlike the training, we don't have the target output in the inference
- We need to feed the model with the previous character and get the next character as the output
  - We have to \"sample\" the next character from the model.
    - For this, we can use the ```torch.multinomial``` function
    - ```torch.multinomial(logits, num_samples=1)```
    - This function will sample the next character from the logits
    - We can use this function to sample the next character in the inference loop
